## Using uv with Docker

This notebook walks through how to combine `uv` with Docker to build, run, and optimize Python containers.

### Outline
- Getting started: run `uv` in a container
- Available images and choosing a base
- Using the pip interface: system vs virtualenv
- Developing in a container: mounts and watch
- Optimizations: caching, intermediate layers
- Non-editable installs and multi-stage builds
- Verifying image provenance
- Practical tips and gotchas


### Why uv + Docker?
**uv** is a fast Python package manager and environment tool. **Docker** provides reproducible containers. Combined, they give:
- **Speed**: uv’s resolver and installer are very fast; Docker caches layers.
- **Reproducibility**: lock files (`uv.lock`) and immutable image layers align well.
- **Smaller images**: multi-stage builds let you keep only the synced virtualenv in the final image.
- **Deterministic builds**: `uv sync --locked` ensures dependencies match the lock.

When building containers, uv can do dependency resolution at build time, while Docker caches results across builds. This keeps runtime images minimal and startup fast.
Reference: [Using uv in Docker](https://docs.astral.sh/uv/guides/integration/docker/)



### Docker layers explained
- **What is a layer?** Each Dockerfile instruction (FROM, RUN, COPY, ADD, etc.) creates an immutable filesystem snapshot called a layer. An image is a stack of read-only layers. A running container adds a thin, writable layer on top.

- **Why they matter**
  - **Caching**: Docker reuses layers when the instruction and its inputs (like files in COPY) are unchanged, speeding up rebuilds.
  - **Order matters**: Place stable steps (e.g., dependency install) before frequently changing steps (e.g., copying app source) to maximize cache hits.
  - **Size and security**: Fewer, smaller final layers reduce push/pull times and shrink the attack surface.
  - **Reuse**: Layers are content-addressed; different images can share them, speeding pulls across projects.

- **Runtime behavior**
  - Containers don’t modify image layers. They write to a top writable layer via copy-on-write. When the container exits, that layer is discarded unless you use volumes.

- **Cache mounts vs layers**
  - **Layers**: Persist in the image/registry; form your image history.
  - **BuildKit cache mounts** (e.g., `--mount=type=cache,target=/root/.cache/uv`): Build-time caches that speed repeated installs (uv downloads, compiled artifacts) without increasing the final image size.

- **Practical uv pattern**
  - Early layer: bind `pyproject.toml` and `uv.lock`, then `uv sync --locked --no-install-project` to cache transitive deps.
  - Later: `ADD .` and `uv sync --locked` to finalize the environment.
  - Final: copy only `.venv` into a clean base and run `uvicorn` or your console script.


### Key concepts: multi-stage builds, caching, and cache mounts
- **Multi-stage builds**: Split the Dockerfile into stages (e.g., `builder` and `runtime`).
  - In the `builder`, run `uv sync` (and compile wheels, etc.).
  - In the `runtime`, copy only the minimal artifacts (e.g., `.venv`) into a clean base, leaving dev tools behind.
  - Benefit: Smaller attack surface, smaller images, faster pulls.

- **Layer caching**: Docker caches each instruction’s resulting layer.
  - If `pyproject.toml` and `uv.lock` don’t change, the layer that installed dependencies remains cached.
  - By copying lock and manifest files before source, we avoid invalidating dependency layers on code changes.

- **Cache mounts**: Ephemeral build-time caches that don’t persist in the image but speed up repeated builds.
  - Example: cache uv’s downloads and Python installations during `RUN` steps.
  - Syntax: `RUN --mount=type=cache,target=/root/.cache/uv uv sync`

Example snippet combining these:
```dockerfile
FROM python:3.12-slim AS builder
COPY --from=ghcr.io/astral-sh/uv:latest /uv /uvx /bin/
WORKDIR /app

# Use lock and manifest to create a stable dependency layer
RUN --mount=type=cache,target=/root/.cache/uv \
    --mount=type=bind,source=uv.lock,target=uv.lock \
    --mount=type=bind,source=pyproject.toml,target=pyproject.toml \
    uv sync --locked --no-install-project --no-editable

# Copy the project and finalize the environment
ADD . /app
RUN --mount=type=cache,target=/root/.cache/uv \
    uv sync --locked --no-editable

FROM python:3.12-slim
COPY --from=builder --chown=app:app /app/.venv /app/.venv
CMD ["/app/.venv/bin/serve", "--host", "0.0.0.0", "--port", "8000"]
```
References:
- Multi-stage builds and non-editable installs: [Using uv in Docker](https://docs.astral.sh/uv/guides/integration/docker/)
- Optimizations > Caching and Intermediate layers: [Guide](https://docs.astral.sh/uv/guides/integration/docker/)



### Prerequisites
- **Docker**: Installed and running.
- **uv** knowledge**:** Basic familiarity with `uv` commands (`uv`, `uv pip`, `uv sync`).
- **A sample Python project** (optional) with `pyproject.toml` and/or `uv.lock`.

We'll use official images and commands from the `uv` docs: [Using uv in Docker](https://docs.astral.sh/uv/guides/integration/docker/).


### Getting started: run `uv` in a container
You can run `uv` from a prebuilt image. For example, a Debian-based image:

```bash
# Show uv help from a Debian-based image
docker run --rm -it ghcr.io/astral-sh/uv:debian uv --help
```
Reference: [Available images and getting started](https://docs.astral.sh/uv/guides/integration/docker/).



### Choosing a base: Available images
`uv` provides both distroless and derived images. Common options:

- **Distroless (uv binaries only)**: `ghcr.io/astral-sh/uv:latest` or versioned tags like `ghcr.io/astral-sh/uv:0.9.2`
- **Alpine-based**: `ghcr.io/astral-sh/uv:alpine` (3.22) or `:alpine3.21`
- **Debian-based**: `ghcr.io/astral-sh/uv:debian`, `:bookworm`, `:trixie`, or `:*-slim`
- **Python-based**: `ghcr.io/astral-sh/uv:python3.12-slim` (and many versions)

Use versioned tags for reproducibility. See the full list in the docs: [Available images](https://docs.astral.sh/uv/guides/integration/docker/).


### Optimizations: caching and intermediate layers
Using Docker’s cache mounts and separating dependency resolution from project code can significantly speed up builds.

- **Cache uv’s downloads** by mounting a cache dir:
```dockerfile
# Example: cache uv downloads and Python installations
RUN --mount=type=cache,target=/root/.cache/uv \
    uv sync
```

- **Pin the cache directories** (optional) to a constant location for reproducibility:
```dockerfile
ENV UV_CACHE_DIR=/opt/uv-cache/
```

- **Install transitive dependencies in a separate layer** using `--no-install-project` so that changes to your app code don’t invalidate dependency layers:
```dockerfile
# Install uv
FROM python:3.12-slim
COPY --from=ghcr.io/astral-sh/uv:latest /uv /uvx /bin/

# Work in /app
WORKDIR /app

# Install only dependencies (not the project)
RUN --mount=type=cache,target=/root/.cache/uv \
    --mount=type=bind,source=uv.lock,target=uv.lock \
    --mount=type=bind,source=pyproject.toml,target=pyproject.toml \
    uv sync --locked --no-install-project

# Now copy the full project
ADD . /app

# Sync the project environment
RUN --mount=type=cache,target=/root/.cache/uv \
    uv sync --locked
```


### Non-editable installs and multi-stage builds
Use `--no-editable` to bake your project into the environment, then copy only the environment to the final image.

```dockerfile
# Stage 1: builder
FROM python:3.12-slim AS builder
COPY --from=ghcr.io/astral-sh/uv:latest /uv /uvx /bin/
WORKDIR /app

# Install transitive dependencies non-editably
RUN --mount=type=cache,target=/root/.cache/uv \
    --mount=type=bind,source=uv.lock,target=uv.lock \
    --mount=type=bind,source=pyproject.toml,target=pyproject.toml \
    uv sync --locked --no-install-project --no-editable

# Add project source and finalize the environment
ADD . /app
RUN --mount=type=cache,target=/root/.cache/uv \
    uv sync --locked --no-editable

# Stage 2: runtime
FROM python:3.12-slim

# Copy only the environment (smaller final image, no source code required at runtime)
COPY --from=builder --chown=app:app /app/.venv /app/.venv

# Run your entrypoint from the venv
CMD ["/app/.venv/bin/serve", "--host", "0.0.0.0", "--port", "8000"]
```


### Verifying image provenance and practical tips
`uv` Docker images are signed; you can verify provenance using GitHub attestations and `cosign`.

- **Verify with GitHub CLI**:
```bash
# Replace :latest with a specific version for best practice, e.g., :0.9.2
gh attestation verify --owner astral-sh oci://ghcr.io/astral-sh/uv:latest
```

- **Alternative verification using cosign**:
```bash
REPO=astral-sh/uv
# Download attestations for the tag (prefer exact version or digest)
gh attestation download --repo $REPO oci://ghcr.io/${REPO}:latest
# Inspect manifest and verify
docker buildx imagetools inspect ghcr.io/${REPO}:latest --format "{{json .Manifest}}" > manifest.json
cosign verify-blob-attestation \
  --new-bundle-format \
  --bundle "$(jq -r .digest manifest.json).jsonl" \
  --certificate-oidc-issuer="https://token.actions.githubusercontent.com" \
  --certificate-identity-regexp="^https://github\\.com/${REPO}/.*" \
  <(jq -j '.|del(.digest,.size)' manifest.json)
```

- **Tips**:
  - Prefer versioned tags or digests over `:latest`.
  - Consider `--no-editable` with multi-stage to slim runtime images.
  - Use cache mounts for `uv` (`/root/.cache/uv`) and optionally set `UV_CACHE_DIR`.
  - For workspaces, use `--no-install-workspace` when appropriate.



### Example app: FastAPI + uv + Docker
This repository now includes an example FastAPI project in `example_fastapi/` with a `pyproject.toml`, a small app, and a multi-stage Dockerfile using `uv`.

- **Project layout**:
  - `example_fastapi/pyproject.toml` — project metadata, dependencies, and `serve` entrypoint (console script)
  - `example_fastapi/src/example_fastapi/app.py` — FastAPI app with `/` and `/healthz`, plus a `run()` entrypoint used by the console script
  - `example_fastapi/Dockerfile` — multi-stage build using `uv` and non-editable install
  - `example_fastapi/.dockerignore`

- **Build the image**:
```bash
cd example_fastapi
# Optionally, use a build kit cache to speed up uv downloads
DOCKER_BUILDKIT=1 docker build -t uv-fastapi-example .
```

- **Run the container**:
```bash
docker run --rm -p 8000:8000 uv-fastapi-example
```

- **Entrypoint note**:
The container uses the venv-installed script path (`/app/.venv/bin/serve`) to run the app, aligning with the non-editable multi-stage pattern in the docs.

- **Test the endpoints**:
```bash
curl http://localhost:8000/
# {"message": "Hello from uv + Docker + FastAPI"}

curl http://localhost:8000/healthz
# {"status": "ok"}
```
This example follows the guidance from the docs for non-editable installs and caching layers. See:
- [Using uv in Docker](https://docs.astral.sh/uv/guides/integration/docker/)
- [Optimizations > Intermediate layers](https://docs.astral.sh/uv/guides/integration/docker/)
- [Non-editable installs](https://docs.astral.sh/uv/guides/integration/docker/)



### docker-compose: develop.watch with uv
The `example_fastapi/docker-compose.yml` uses Compose’s `develop.watch` to rebuild when `pyproject.toml`, `uv.lock`, or `src/` change. This matches the non-editable image pattern (we rebuild the image rather than hot-reloading code inside).

```yaml
services:
  app:
    build:
      context: .
    ports:
      - "8000:8000"
    command: ["/app/.venv/bin/serve", "--host", "0.0.0.0", "--port", "8000"]
    develop:
      watch:
        - action: rebuild
          path: ./pyproject.toml
        - action: rebuild
          path: ./uv.lock
        - action: rebuild
          path: ./src
```

Run with:
```bash
cd example_fastapi
docker compose up --build
```

Reference: [Developing in a container > Configuring watch with docker compose](https://docs.astral.sh/uv/guides/integration/docker/).


### pyproject.toml: build-system and console script
To generate a venv console script and support non-editable installs, set a wheel build backend and define a script entrypoint.

```toml
[build-system]
requires = ["hatchling>=1.25"]
build-backend = "hatchling.build"

[project]
name = "example-fastapi-app"
version = "0.1.0"
requires-python = ">=3.10"
dependencies = [
  "fastapi>=0.115,<1",
  "uvicorn[standard]>=0.30,<1",
]

[project.scripts]
serve = "example_fastapi.app:run"

[tool.hatch.build.targets.wheel]
packages = ["src/example_fastapi"]
```

This ensures `uv sync --no-editable` can build and install your project, and `/app/.venv/bin/serve` will be available in the image.